# Mosaic integration: [RNA, RNA + ADT, ADT]

Mosaic integration combines batches that share only some modalities. This tutorial is for an RNA batch, an RNA + ADT batch and an ADT batch. It runs StabMap and scMoMaT on `D38mini`, 6,000 cells of the benchmark dataset `D38`, and then on data in your own format.

Methods run on Linux. On macOS or Windows, [open this notebook in Colab](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_mosaic_rna_adt.ipynb).

## 1. Install

This cell installs `multibench-sc`. It needs no restart. On Colab, choose a T4 GPU runtime first (Runtime -> Change runtime type), because changing it later loses the downloads.

In [ ]:
import numpy, pandas
pins = f"numpy=={numpy.__version__} pandas=={pandas.__version__}"
%pip install -q "multibench-sc>=0.5.0" {pins}

## 2. Download the data and the environments

`mtb.data.fetch` downloads `D38mini` (2 MB) once. `mtb.env.install` downloads the environments for StabMap and scMoMaT. The download is 1.8 GB on a computer without a GPU and 5.4 GB on one with a GPU.

In [ ]:
import multibench as mtb

mtb.data.fetch("D38mini")
METHODS = ["StabMap", "scMoMaT"]
envs = mtb.env.install(METHODS)

## 3. Run the methods

`run` runs each method on `D38mini` and scores its output with the scIB metrics. The clustering metrics score how well the embedding's clusters and neighbourhoods match the cell types, and how separate the clusters are. The batch metrics measure how well the batches mix. The classification scores measure how well the cell types learned on one batch are predicted in the other batches. scMoMaT writes a graph, so it gets no classification scores. `res.summary` has one row per method with its status, run time and scores. Higher is better for every metric.

<details>
<summary>Details</summary>

scMoMaT writes a graph instead of an embedding. `run` scores its UMAP.

`params={"Method": {"key": value}}` sets a method's parameters, and `mtb.params_for` lists them. Many methods take none.

`mtb.load_run("out/D38mini")` reloads these results later without running anything.

</details>

In [ ]:
res = mtb.run("D38mini", "mosaic", methods=METHODS,
                  out_dir="out/D38mini")
res.summary

## 4. Plot

`res.plot()` draws the scores as a bubble table. Each column is scaled on its own: its lightest fill is that column's lowest value, not zero.

<details>
<summary>Details</summary>

Metrics are grouped by family: blue for clustering, green for batch correction, pink for classification. Each family starts with an `Overall` bar: each row's mean rank over the family's metrics, scaled so that the lowest gets the shortest bar.

A column whose rows all hold the same value is drawn grey, and a warning names it. A figure of one method is grey everywhere.

</details>

In [ ]:
res.plot()

## 5. Your own data

Your data needs raw counts, with one `.h5ad` file per batch and modality and a cell type for each cell. `mtb.io.export_dataset` with `batch_index=` writes one batch per call. Number the batches as below: RNA, then RNA + ADT, then ADT.

Every RNA file needs the same gene names and every ADT file the same protein names, in the same order. scMoMaT matches the features of two batches by position, and the check does not compare their names across batches.

Starting from something else? See the recipes for [a 10x `.h5` file](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-10x-h5), [labels in a CSV file](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-labels-csv) and [no labels](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-no-labels).

<details>
<summary>Details: export</summary>

The names in quotes say where your file holds each part. `"X"` is the main matrix, `"obsm:protein"` the matrix `obsm["protein"]`, `"layer:counts"` the layer `layers["counts"]`, and `"obs:celltype"` the column `obs["celltype"]`, which `"celltype"` alone names too. A wrong name after `obsm:`, `layer:` or `obs:` raises an error that lists the names the file has.

[The folder layout](https://dsichang.github.io/scMultiBench/tutorials/run/#folder-layout) lists the batch patterns and the methods that accept each one.

When the ADT file also holds a matrix in `obsm` under a name that does not start with `X_`, `export_dataset` asks which one is the protein matrix. `X_umap` and `X_pca` are left alone.

Without `methods=`, `scan` lists every method that reads the modalities in the folder.

</details>

So that the notebook runs as it is, the next cell writes four small example files, one per batch and modality, from the first 1,500 cells of each batch of `D38mini`.

In [ ]:
mtb.data.write_example("D38mini")

Read your files. For your own data, change every file name marked `# <-`. On Colab, upload the files first: click the folder icon on the left, then the upload icon.

In [ ]:
import anndata as ad

rna1 = ad.read_h5ad("my_rna1.h5ad")   # <- batch 1: your RNA file
rna2 = ad.read_h5ad("my_rna2.h5ad")   # <- batch 2: your RNA file
adt2 = ad.read_h5ad("my_adt2.h5ad")   # <- batch 2: proteins (ADT), same cells
adt3 = ad.read_h5ad("my_adt3.h5ad")   # <- batch 3: your protein (ADT) file
print(rna2)
print(adt2)

Write the dataset folder. Change each name in quotes marked `# <-`. The printout above lists the names in two of your files. Write a name listed after `obs:` with that prefix, as `"obs:celltype"`.

In [ ]:
NAME = "MYMOSAIC_RNA_ADT"   # <- a new name for your own data
out = "mydata/" + NAME
celltype = "obs:celltype"   # <- the cell-type column of every file
# batch 1: RNA only
mtb.io.export_dataset(rna1, out, rna="X", labels=celltype,
                      batch_index=1, category="mosaic", overwrite=True)
# batch 2: RNA + ADT (one file: adt="obsm:protein")
mtb.io.export_dataset(rna2, out, rna="X", adt=adt2, labels=celltype,
                      batch_index=2, category="mosaic", overwrite=True)
# batch 3: ADT only
mtb.io.export_dataset(adt3, out, rna=None, adt="X", labels=celltype,
                      batch_index=3, category="mosaic", overwrite=True)
mtb.scan("mydata/" + NAME, "mosaic", methods=METHODS)

Run the same methods on the folder:

In [ ]:
mine = mtb.run("mydata/" + NAME, "mosaic", methods=METHODS,
                   out_dir="out/" + NAME)
mine.summary

In [ ]:
mine.plot()

## 6. Every method of this task

3 methods run this task. [This notebook](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_mosaic_rna_adt_all.ipynb) runs them all. It downloads 3 envs, 6.2 GB on a computer without a GPU and 5.9 GB on one with a GPU.

## Troubleshooting

`res.summary[res.summary.status != "scored"]` lists each method that failed or was skipped. Its `reason` column ends with the method's error output.

<details>
<summary>Details</summary>

| symptom | fix |
|---|---|
| a method fails on a small dataset | `mtb.scan` names what the method needs from the data's size in its `caveat` column |
| a warning that values are not whole numbers | export raw counts, for example with `rna="layer:counts"` |
| `... matrix/data as cells x features` | the matrix is transposed: export it again with `mtb.io.export_dataset` |
| a method times out | raise `timeout=` in `run` |
| batch metrics use the wrong batches | `res.rescore(batch=my_vector)` scores again without running the methods |

</details>

## Next steps

- The guides: [run](https://dsichang.github.io/scMultiBench/tutorials/run/), [evaluate](https://dsichang.github.io/scMultiBench/tutorials/evaluate/), [plot](https://dsichang.github.io/scMultiBench/tutorials/plot/) and [discover methods](https://dsichang.github.io/scMultiBench/tutorials/discover/).
- The [interactive explorer](https://shiny.maths.usyd.edu.au/scMultiBench/) has the full benchmark's rankings, with no install needed.